<!-- field-paper-role-banner -->
> 🟩 **정답·해설본** — 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 수식과 tensor shape를 설명하세요.

# 컴퓨터 비전 · 06. Faster R-CNN

## Faster R-CNN: Towards Real-Time Object Detection with Region Proposal Networks

- **원 논문:** [Faster R-CNN: Towards Real-Time Object Detection with Region Proposal Networks](https://arxiv.org/abs/1506.01497)
- **저자 / 발표:** Shaoqing Ren, Kaiming He, Ross Girshick, Jian Sun · NeurIPS 2015 (2015)
- **난이도 / 예상 시간:** 중급 · 약 120분
- **선수 지식:** Python, PyTorch tensor, 합성곱, 역전파의 기초
- **로컬 학습 데이터:** `data/field_curriculum/vision_shapes.npz`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** RPN anchor, IoU assignment, objectness+box multi-task loss를 재현한다.

**축소하거나 생략한 부분:** VGG-16, 9 anchors/location, VOC/COCO 대신 4×4 단일-scale anchor grid를 쓴다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §3.1 Fig. 3: sliding RPN | Task 1 anchor grid | 16 anchors |
| §3.1.1: anchors | Task 1 IoU | best anchor |
| §3.1.2 Eq. (1): Lcls+λLreg | Task 2 | finite loss |
| §3.1.2 Eq. (2): box parameterization | Task 1 delta | identity delta |

## 핵심 아이디어와 수식

$$L=\frac{1}{N_{cls}}\sum_iL_{cls}(p_i,p_i^*)+\lambda\frac{1}{N_{reg}}\sum_i p_i^*L_{reg}(t_i,t_i^*)$$

**기호 해설:** p는 objectness, p*는 anchor label, t는 box delta, λ는 regression 가중치다.

**코드 대응:** Task 1이 anchor/IoU/box encoding, Task 2가 RPNHead, Task 3이 multi-task loss를 맡는다.

**입력과 출력 shape:** 입력 [B,1,16,16] → feature [B,12,4,4] → object [B,16], delta [B,16,4].

**포트폴리오 구현 범위:** 두 번째 Fast R-CNN stage와 NMS는 제외하고 논문의 RPN 학습 경로를 깊게 구현한다.

아래 코드는 결과만 내는 라이브러리 호출로 핵심을 감추지 않는다. config, 논문 고유
연산, 모델의 `forward`, 목적함수, `train_step`, 평가 함수를 순서대로 직접 작성한다.

In [ ]:
from dataclasses import dataclass
from pathlib import Path
import itertools
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(20260814)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device


def locate(relative_path):
    for base in (Path.cwd(), *Path.cwd().parents):
        candidate = base / relative_path
        if candidate.exists():
            return candidate
    raise FileNotFoundError(relative_path)


dataset_path = locate("data/field_curriculum/vision_shapes.npz")
raw = np.load(dataset_path)
images = torch.from_numpy(raw["images"]).float()
labels = torch.from_numpy(raw["labels"]).long()
masks = torch.from_numpy(raw["masks"]).float()
boxes = torch.from_numpy(raw["boxes"]).float()
train_idx = torch.from_numpy(raw["train_idx"][:128]).long()
test_idx = torch.from_numpy(raw["test_idx"][:64]).long()
images, labels, masks, boxes = ACCELERATOR.move(
    images,
    labels,
    masks,
    boxes,
)
train_idx, test_idx = ACCELERATOR.move(train_idx, test_idx)
class_names = raw["class_names"].tolist()
x_train = images[train_idx]
y_train = labels[train_idx]
assert images.shape == (480, 1, 16, 16)
assert masks.shape == (480, 16, 16)
assert boxes.shape == (480, 4)
print(ACCELERATOR.summary())
print(dataset_path.name, images.shape, class_names)

### 구현 단계 1 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §3.1 Fig. 3: sliding RPN
- **노트북에서 구현할 부분:** Task 1 anchor grid
- **구현 이유:** 논문의 계산 경로를 작은 tensor로 보존하면서 각 중간값을
  assertion으로 관찰하기 위해서다.
- **완료 증거:** 16 anchors

먼저 입력 shape와 기대 출력 shape를 종이에 적은 뒤 아래 TODO를 구현한다.

In [ ]:
@dataclass
class RPNConfig:
    grid_size: int = 4
    box_size: float = 0.35
    feature_channels: int = 12
    learning_rate: float = 0.02
    steps: int = 18
    regression_weight: float = 1.0


def generate_anchors(grid_size, box_size, device):
    centers = torch.linspace(
        0.5 / grid_size,
        1.0 - 0.5 / grid_size,
        grid_size,
        device=device,
    )
    grid_y, grid_x = torch.meshgrid(centers, centers, indexing="ij")
    half = box_size / 2
    anchors = torch.stack(
        (grid_x - half, grid_y - half, grid_x + half, grid_y + half),
        dim=-1,
    )
    return anchors.reshape(-1, 4).clamp(0.0, 1.0)


def pairwise_iou(first, second):
    left_top = torch.maximum(first[:, None, :2], second[None, :, :2])
    right_bottom = torch.minimum(first[:, None, 2:], second[None, :, 2:])
    intersection = (right_bottom - left_top).clamp_min(0).prod(dim=-1)
    first_area = (first[:, 2:] - first[:, :2]).prod(dim=-1)[:, None]
    second_area = (second[:, 2:] - second[:, :2]).prod(dim=-1)[None]
    union = first_area + second_area - intersection
    return intersection / union.clamp_min(1e-8)


def encode_box(anchor, target):
    anchor_center = (anchor[:2] + anchor[2:]) / 2
    anchor_size = anchor[2:] - anchor[:2]
    target_center = (target[:2] + target[2:]) / 2
    target_size = target[2:] - target[:2]
    center_delta = (target_center - anchor_center) / anchor_size
    size_delta = torch.log(target_size / anchor_size)
    return torch.cat((center_delta, size_delta))


config = RPNConfig()
anchors = generate_anchors(config.grid_size, config.box_size, DEVICE)
assert anchors.shape == (16, 4)
assert torch.allclose(encode_box(anchors[0], anchors[0]), anchors.new_zeros(4))

### 구현 단계 2 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §3.1.1: anchors
- **노트북에서 구현할 부분:** Task 1 IoU
- **구현 이유:** 논문의 계산 경로를 작은 tensor로 보존하면서 각 중간값을
  assertion으로 관찰하기 위해서다.
- **완료 증거:** best anchor

먼저 입력 shape와 기대 출력 shape를 종이에 적은 뒤 아래 TODO를 구현한다.

In [ ]:
class RegionProposalNetwork(nn.Module):
    def __init__(self, config):
        super().__init__()
        channels = config.feature_channels
        self.backbone = nn.Sequential(
            nn.Conv2d(1, channels, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.shared = nn.Conv2d(channels, channels, 3, padding=1)
        self.objectness = nn.Conv2d(channels, 1, kernel_size=1)
        self.box_regression = nn.Conv2d(channels, 4, kernel_size=1)

    def forward(self, inputs):
        features = F.relu(self.shared(self.backbone(inputs)))
        object_logits = self.objectness(features).flatten(start_dim=1)
        box_deltas = self.box_regression(features)
        box_deltas = box_deltas.permute(0, 2, 3, 1).reshape(len(inputs), -1, 4)
        return object_logits, box_deltas


model = RegionProposalNetwork(config).to(DEVICE)
probe_object, probe_delta = model(images[:2])
assert probe_object.shape == (2, 16)
assert probe_delta.shape == (2, 16, 4)

### 구현 단계 3 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §3.1.2 Eq. (2): box parameterization
- **노트북에서 구현할 부분:** Task 1 delta
- **구현 이유:** 논문의 계산 경로를 작은 tensor로 보존하면서 각 중간값을
  assertion으로 관찰하기 위해서다.
- **완료 증거:** identity delta

먼저 입력 shape와 기대 출력 shape를 종이에 적은 뒤 아래 TODO를 구현한다.

In [ ]:
def build_rpn_targets(anchors, target_boxes):
    object_targets = []
    delta_targets = []
    for target_box in target_boxes:
        overlaps = pairwise_iou(anchors, target_box[None]).squeeze(1)
        positive_index = int(overlaps.argmax())
        object_target = torch.zeros(len(anchors), device=anchors.device)
        object_target[positive_index] = 1.0
        delta_target = torch.zeros(len(anchors), 4, device=anchors.device)
        delta_target[positive_index] = encode_box(
            anchors[positive_index],
            target_box,
        )
        object_targets.append(object_target)
        delta_targets.append(delta_target)
    return torch.stack(object_targets), torch.stack(delta_targets)


def rpn_loss(prediction, targets, config):
    object_logits, box_deltas = prediction
    object_targets, delta_targets = targets
    classification = F.binary_cross_entropy_with_logits(
        object_logits,
        object_targets,
    )
    positive = object_targets.bool()
    regression = F.smooth_l1_loss(
        box_deltas[positive],
        delta_targets[positive],
    )
    return classification + config.regression_weight * regression


def train_rpn_step(model, optimizer, inputs, targets, config):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    loss = rpn_loss(model(inputs), targets, config)
    loss.backward()
    optimizer.step()
    return float(loss.detach())


def evaluate_rpn(model, inputs, object_targets):
    model.eval()
    with torch.no_grad():
        object_logits, _ = model(inputs)
    predicted = object_logits.argmax(dim=1)
    expected = object_targets.argmax(dim=1)
    return float((predicted == expected).float().mean())


targets = build_rpn_targets(anchors, boxes[train_idx])
optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)
losses = []
for step in range(config.steps):
    loss = train_rpn_step(model, optimizer, x_train, targets, config)
    losses.append(loss)
test_targets = build_rpn_targets(anchors, boxes[test_idx])
recall = evaluate_rpn(model, images[test_idx], test_targets[0])
assert min(losses[3:]) < losses[0]
assert 0.0 <= recall <= 1.0
plt.figure(figsize=(5, 3))
plt.plot(losses)
plt.title(f"RPN best-anchor recall={recall:.1%}")
plt.tight_layout()
plt.show()

## 정답 해석 가이드

핵심은 최종 숫자를 외우는 것이 아니라 **원문 위치 → 수식 → 코드 → 반증 가능한 증거**를
연결하는 것입니다. 이 노트북은 다음 원 규모 조건을 재현하지 않았습니다:

> VGG-16, 9 anchors/location, VOC/COCO 대신 4×4 단일-scale anchor grid를 쓴다.

다음 실험에서는 데이터 크기, 모델 폭, 학습 step 중 하나만 바꾸고 동일 seed에서 비교하세요.